In [ ]:
import time
import math
import csv
import numpy as np
import board
import busio
import sys
import signal
import select
import threading
import cv2
import os

import adafruit_ads1x15.ads1115 as ADS
from adafruit_ads1x15.analog_in import AnalogIn
from ultralytics import YOLO
from evdev import InputDevice, ecodes

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt


# =========================================================
#                 ADC + MICS
# =========================================================
i2c = busio.I2C(board.SCL, board.SDA)

ads1 = ADS.ADS1115(i2c, address=0x48)
ads2 = ADS.ADS1115(i2c, address=0x49)

ads1.gain = 1
ads2.gain = 1
ads1.data_rate = 860
ads2.data_rate = 860

try:
    ads1.mode = ADS.Mode.CONTINUOUS
    ads2.mode = ADS.Mode.CONTINUOUS
except Exception:
    pass

mic1 = AnalogIn(ads1, 0)
mic2 = AnalogIn(ads2, 0)

MIC_SAMPLES_PER_TICK = 120


def measure_db_fast(channel, n_samples=MIC_SAMPLES_PER_TICK):
    s = 0.0
    ss = 0.0

    for _ in range(n_samples):
        x = float(channel.value)
        s += x
        ss += x * x

    mean = s / n_samples
    var = (ss / n_samples) - (mean * mean)

    if var < 1e-9:
        return -120.0, 0.0

    rms = math.sqrt(var)
    dbfs = 20.0 * math.log10(rms / 16384.0)

    return dbfs, rms


# =========================================================
#                 USB CAMERA / YOLO SETTINGS
# =========================================================
MODEL_NAME = "yolov8n.pt"
CONF_THRESH = 0.4
IOU_THRESH = 0.45
PERSON_CLASS_ID = 0

SHOW_WINDOW = True

# CHANGE THIS after running:
# v4l2-ctl --list-devices
# Example: if USB camera says /dev/video36, keep "/dev/video36".
USB_CAMERA_DEVICE = "/dev/video36"

CAM_SIZE = (320, 240)
USB_CAMERA_FPS = 30
USB_CAMERA_FOURCC = "MJPG"

YOLO_FPS = 10.0
YOLO_PERIOD = 1.0 / YOLO_FPS

MAX_PERSON_COUNT = 2


# =========================================================
#                 OUTPUT FILES
# =========================================================
CSV_FILENAME = "Final_Stomping_Multiclass_usb.csv"
PLOT_FILENAME = "Final_graph_multiclass_usb.png"
OVERLAY_PLOT_FILENAME = "Overlay_graph_multiclass_usb.png"
CAM_HISTORY_PLOT_FILENAME = "Camera_history_multiclass_usb.png"


# =========================================================
#                 BUFFERS
# =========================================================
rms1_values, db1_values = [], []
rms2_values, db2_values = [], []

time_sec_values = []
human_count_values = []
mouse_event_values = []

cam_hist_lock = threading.Lock()
cam_hist_time = []
cam_hist_count = []

START_TIME = None


# =========================================================
#                 SHARED FLAGS / LOCKS
# =========================================================
stop_flag = False

cam_count_lock = threading.Lock()
cam_max_count_this_second = 0

mouse_lock = threading.Lock()
mouse_event_value = 0

MOUSE_EVENT_PATH = "/dev/input/by-id/usb-YICHIP_Wireless_Device-if01-event-mouse"


# =========================================================
#                 FINALIZATION / PLOTS
# =========================================================
def finalize_csv():
    if len(rms1_values) == 0:
        return

    sd_rms1 = float(np.std(rms1_values, ddof=0))
    sd_db1 = float(np.std(db1_values, ddof=0))
    sd_rms2 = float(np.std(rms2_values, ddof=0))
    sd_db2 = float(np.std(db2_values, ddof=0))

    with open(CSV_FILENAME, mode="a", newline="") as f:
        w = csv.writer(f)
        w.writerow([])
        w.writerow(["==== FINAL STANDARD DEVIATIONS ===="])
        w.writerow(["Mic1 RMS SD", sd_rms1])
        w.writerow(["Mic1 dBFS SD", sd_db1])
        w.writerow(["Mic2 RMS SD", sd_rms2])
        w.writerow(["Mic2 dBFS SD", sd_db2])

    print("\n========= FINAL STATISTICS =========")
    print(f"Total samples: {len(rms1_values)}")
    print(f"Mic1 RMS SD:   {sd_rms1:.4f}")
    print(f"Mic1 dBFS SD:  {sd_db1:.4f}")
    print(f"Mic2 RMS SD:   {sd_rms2:.4f}")
    print(f"Mic2 dBFS SD:  {sd_db2:.4f}")
    print("===================================\n")


def save_subplot_figure():
    if len(time_sec_values) == 0:
        return

    t = np.array(time_sec_values, dtype=float)
    y1 = np.array(rms1_values, dtype=float)
    y2 = np.array(rms2_values, dtype=float)
    yh = np.array(human_count_values, dtype=float)
    ym = np.array(mouse_event_values, dtype=float)

    fig, axs = plt.subplots(4, 1, figsize=(12, 10), sharex=True)

    axs[0].plot(t, y1, linewidth=1.5)
    axs[0].set_title("Mic 1 RMS")
    axs[0].set_ylabel("RMS")
    axs[0].grid(True, alpha=0.25)

    axs[1].plot(t, y2, linewidth=1.5)
    axs[1].set_title("Mic 2 RMS")
    axs[1].set_ylabel("RMS")
    axs[1].grid(True, alpha=0.25)

    axs[2].step(t, yh, where="post", linewidth=2.0)
    axs[2].set_title("YOLO Human Count")
    axs[2].set_ylabel("Human Count")
    axs[2].set_ylim(-0.2, 2.2)
    axs[2].set_yticks([0, 1, 2])
    axs[2].grid(True, alpha=0.25)

    axs[3].step(t, ym, where="post", linewidth=2.0)
    axs[3].set_title("Mouse Event: Left=1, Right=2")
    axs[3].set_ylabel("Mouse Event")
    axs[3].set_ylim(-0.2, 2.2)
    axs[3].set_yticks([0, 1, 2])
    axs[3].set_xlabel("Time (seconds)")
    axs[3].grid(True, alpha=0.25)

    plt.tight_layout()
    fig.savefig(PLOT_FILENAME, dpi=300)
    plt.close(fig)

    print(f"[PLOT] Saved subplot figure to: {PLOT_FILENAME}")


def save_overlay_figure():
    if len(time_sec_values) == 0:
        return

    t = np.array(time_sec_values, dtype=float)
    y1 = np.array(rms1_values, dtype=float)
    y2 = np.array(rms2_values, dtype=float)
    yh = np.array(human_count_values, dtype=float)
    ym = np.array(mouse_event_values, dtype=float)

    fig, ax1 = plt.subplots(figsize=(12, 5))

    l1, = ax1.plot(t, y1, label="Mic1 RMS", linewidth=1.8)
    l2, = ax1.plot(t, y2, label="Mic2 RMS", linewidth=1.8)

    ax1.set_xlabel("Time (seconds)")
    ax1.set_ylabel("RMS")
    ax1.grid(True, alpha=0.25)

    ax2 = ax1.twinx()
    l3 = ax2.step(t, yh, where="post", label="YOLO Count", linewidth=2.2)[0]
    l4 = ax2.step(t, ym, where="post", label="Mouse Event", linewidth=2.2)[0]

    ax2.set_ylabel("Count / Event")
    ax2.set_ylim(-0.2, 2.2)
    ax2.set_yticks([0, 1, 2])

    ax1.legend(
        [l1, l2, l3, l4],
        [l1.get_label(), l2.get_label(), l3.get_label(), l4.get_label()],
        loc="upper right"
    )

    ax1.set_title("Overlay: Mic RMS + YOLO Count + Mouse Event")

    plt.tight_layout()
    fig.savefig(OVERLAY_PLOT_FILENAME, dpi=300)
    plt.close(fig)

    print(f"[PLOT] Saved overlay figure to: {OVERLAY_PLOT_FILENAME}")


def save_camera_history_plot():
    with cam_hist_lock:
        if len(cam_hist_time) == 0:
            print("[CAM-HIST] No camera history to plot.")
            return

        t_cam = np.array(cam_hist_time, dtype=float)
        y_cam = np.array(cam_hist_count, dtype=float)

    fig = plt.figure(figsize=(12, 4))
    plt.step(t_cam, y_cam, where="post", linewidth=1.5)
    plt.title(f"Camera Count History (~{YOLO_FPS:.0f} FPS)")
    plt.ylabel("Detected People Count")
    plt.ylim(-0.2, 2.2)
    plt.yticks([0, 1, 2])
    plt.xlabel("Time (seconds)")
    plt.grid(True, alpha=0.25)
    plt.tight_layout()
    fig.savefig(CAM_HISTORY_PLOT_FILENAME, dpi=300)
    plt.close(fig)

    print(f"[CAM-HIST] Saved camera history plot to: {CAM_HISTORY_PLOT_FILENAME}")


# =========================================================
#                 EXIT HANDLING
# =========================================================
def handle_exit(signum, frame):
    global stop_flag
    stop_flag = True
    sys.exit(0)


signal.signal(signal.SIGINT, handle_exit)


# =========================================================
#                 USB CAMERA THREAD
# =========================================================
class VideoCaptureThread(threading.Thread):
    def __init__(self, cam):
        super().__init__(daemon=True)
        self.cam = cam
        self.frame = None
        self.running = True
        self.lock = threading.Lock()

    def run(self):
        while self.running:
            ret, frame = self.cam.read()

            if ret and frame is not None:
                with self.lock:
                    self.frame = frame
            else:
                time.sleep(0.01)

    def get_frame(self):
        with self.lock:
            if self.frame is None:
                return None
            return self.frame.copy()

    def stop(self):
        self.running = False


# =========================================================
#                 CAMERA / YOLO WORKER
# =========================================================
def camera_worker():
    global stop_flag, cam_max_count_this_second, START_TIME

    print("[CAM] Loading YOLO model...")
    model = YOLO(MODEL_NAME)

    print(f"[CAM] Opening USB camera: {USB_CAMERA_DEVICE}")

    cam = cv2.VideoCapture(USB_CAMERA_DEVICE, cv2.CAP_V4L2)

    if not cam.isOpened():
        print(f"[CAM] ERROR: Could not open {USB_CAMERA_DEVICE}")
        print("[CAM] Run: v4l2-ctl --list-devices")
        return

    cam.set(
        cv2.CAP_PROP_FOURCC,
        cv2.VideoWriter_fourcc(*USB_CAMERA_FOURCC)
    )
    cam.set(cv2.CAP_PROP_FRAME_WIDTH, CAM_SIZE[0])
    cam.set(cv2.CAP_PROP_FRAME_HEIGHT, CAM_SIZE[1])
    cam.set(cv2.CAP_PROP_FPS, USB_CAMERA_FPS)

    # Verify that the camera can actually produce frames.
    test_ok = False

    for _ in range(30):
        ret, test_frame = cam.read()

        if ret and test_frame is not None:
            test_ok = True
            print(
                f"[CAM] USB camera ready: "
                f"{test_frame.shape[1]}x{test_frame.shape[0]}"
            )
            break

        time.sleep(0.1)

    if not test_ok:
        print(f"[CAM] ERROR: {USB_CAMERA_DEVICE} opened but cannot read frames.")
        cam.release()
        return

    capture_thread = VideoCaptureThread(cam)
    capture_thread.start()

    next_yolo = time.time()

    try:
        while not stop_flag:
            frame_bgr = capture_thread.get_frame()

            if frame_bgr is None:
                time.sleep(0.01)
                continue

            now = time.time()

            if now < next_yolo:
                time.sleep(max(0.0, next_yolo - now))

            next_yolo += YOLO_PERIOD

            results = model.predict(
                source=frame_bgr,
                conf=CONF_THRESH,
                iou=IOU_THRESH,
                classes=[PERSON_CLASS_ID],
                verbose=False
            )

            boxes = results[0].boxes
            human_count = len(boxes) if boxes is not None else 0
            human_count = min(int(human_count), MAX_PERSON_COUNT)

            if START_TIME is not None:
                with cam_hist_lock:
                    cam_hist_time.append(time.time() - START_TIME)
                    cam_hist_count.append(human_count)

            with cam_count_lock:
                cam_max_count_this_second = max(
                    cam_max_count_this_second,
                    human_count
                )

            if SHOW_WINDOW:
                if boxes is not None:
                    for b in boxes:
                        x1, y1, x2, y2 = map(int, b.xyxy[0].tolist())
                        conf = float(b.conf[0])

                        cv2.rectangle(
                            frame_bgr,
                            (x1, y1),
                            (x2, y2),
                            (0, 255, 0),
                            2
                        )

                        cv2.putText(
                            frame_bgr,
                            f"Person ({conf:.2f})",
                            (x1, max(20, y1 - 10)),
                            cv2.FONT_HERSHEY_SIMPLEX,
                            0.55,
                            (0, 255, 0),
                            2
                        )

                cv2.putText(
                    frame_bgr,
                    f"Human count: {human_count}",
                    (10, 25),
                    cv2.FONT_HERSHEY_SIMPLEX,
                    0.7,
                    (0, 255, 0),
                    2
                )

                cv2.imshow("USB YOLO Human Count", frame_bgr)

                if cv2.waitKey(1) & 0xFF == ord("q"):
                    stop_flag = True
                    break

    except Exception as e:
        print(f"[CAM] ERROR in camera thread: {e}")

    finally:
        capture_thread.stop()
        capture_thread.join(timeout=2.0)

        cam.release()

        if SHOW_WINDOW:
            cv2.destroyAllWindows()

        print("[CAM] USB camera closed.")


# =========================================================
#                 MOUSE THREAD
# =========================================================
def mouse_worker_evdev():
    global mouse_event_value, stop_flag

    try:
        if not os.path.exists(MOUSE_EVENT_PATH):
            print(f"[MOUSE] ERROR: {MOUSE_EVENT_PATH} not found.")
            print("[MOUSE] Run: ls -l /dev/input/by-id/ | grep -i mouse")
            return

        dev = InputDevice(MOUSE_EVENT_PATH)

        print(f"[MOUSE] Listening on {dev.path} | name='{dev.name}'")
        print("[MOUSE] Left click = event 1 | Right click = event 2")

        for event in dev.read_loop():
            if stop_flag:
                break

            if event.type == ecodes.EV_KEY and event.value == 1:
                if event.code == ecodes.BTN_LEFT:
                    with mouse_lock:
                        mouse_event_value = 1
                    print("[MOUSE] Logged event: 1")

                elif event.code == ecodes.BTN_RIGHT:
                    with mouse_lock:
                        mouse_event_value = 2
                    print("[MOUSE] Logged event: 2")

    except PermissionError:
        print("[MOUSE] Permission denied reading input device.")
        print("[MOUSE] Run with sudo or add your user to the input group.")

    except Exception as e:
        print(f"[MOUSE] Unexpected error: {e}")


# =========================================================
#                 MAIN LOGGER
# =========================================================
def main():
    global stop_flag
    global cam_max_count_this_second
    global mouse_event_value
    global START_TIME

    START_TIME = time.time()

    cam_thread = threading.Thread(target=camera_worker, daemon=True)
    cam_thread.start()

    mouse_thread = threading.Thread(target=mouse_worker_evdev, daemon=True)
    mouse_thread.start()

    with open(CSV_FILENAME, mode="w", newline="") as f:
        writer = csv.writer(f)

        writer.writerow([
            "timestamp",
            "RMS_mic1",
            "dBFS_mic1",
            "RMS_mic2",
            "dBFS_mic2",
            "human_count",
            "mouse_event"
        ])

        print("\nLogging started.")
        print("Left click = logs 1 for that second")
        print("Right click = logs 2 for that second")
        print("No click = logs 0")
        print("Type q + Enter or press Ctrl+C to stop.\n")

        next_tick = time.time()
        t0 = None

        try:
            while not stop_flag:
                now = time.time()

                if now < next_tick:
                    time.sleep(next_tick - now)

                tick_start = time.time()
                next_tick += 1.0

                db1, rms1 = measure_db_fast(mic1)
                db2, rms2 = measure_db_fast(mic2)

                rms1_values.append(rms1)
                db1_values.append(db1)

                rms2_values.append(rms2)
                db2_values.append(db2)

                with cam_count_lock:
                    human_this_second = int(cam_max_count_this_second)
                    cam_max_count_this_second = 0

                with mouse_lock:
                    mouse_this_second = int(mouse_event_value)
                    mouse_event_value = 0

                if t0 is None:
                    t0 = tick_start

                time_sec_values.append(int(round(tick_start - t0)))
                human_count_values.append(human_this_second)
                mouse_event_values.append(mouse_this_second)

                timestamp = time.strftime("%Y-%m-%d %H:%M:%S")

                writer.writerow([
                    timestamp,
                    f"{rms1:.4f}",
                    f"{db1:.2f}",
                    f"{rms2:.4f}",
                    f"{db2:.2f}",
                    human_this_second,
                    mouse_this_second
                ])

                f.flush()

                print(
                    f"{timestamp} | "
                    f"Mic1 RMS: {rms1:8.2f} dBFS: {db1:7.2f} | "
                    f"Mic2 RMS: {rms2:8.2f} dBFS: {db2:7.2f} | "
                    f"Camera count: {human_this_second} | "
                    f"Mouse event: {mouse_this_second}"
                )

                if sys.stdin in select.select([sys.stdin], [], [], 0)[0]:
                    line = sys.stdin.readline().strip()

                    if line.lower() == "q":
                        stop_flag = True
                        break

                elapsed = time.time() - tick_start

                if elapsed > 1.0:
                    next_tick = time.time() + 1.0

        finally:
            stop_flag = True

            finalize_csv()
            save_subplot_figure()
            save_overlay_figure()
            save_camera_history_plot()


if __name__ == "__main__":
    main()